<a href="https://colab.research.google.com/github/abubakirovx/JaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install bitsandbytes dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [bitsandbytes]


In [21]:
from transformers import AutoTokenizer, BitsAndBytesConfig , AutoModelForCausalLM
import torch
from datasets import load_dataset
from peft import LoraConfig , get_peft_model


In [3]:


model_id="Qwen/Qwen2.5-3B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model_id)

# print("Vocab_size:",tokenizer.vocab_size)
# print("special tokens_map",tokenizer.special_tokens_map)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"


)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",#GPU CPU
    # dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [4]:
# Before fine-tuning
# prompt="Explain what is tokenizer?"
prompt="How are you tiny?"

inputs = tokenizer(prompt,return_tensors="pt").to(model.device) # GPU CPU

with torch.no_grad():
    output_ids=model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )


print("Answer=>",tokenizer.decode(output_ids[0],skip_special_tokens=True))

print(model)

Answer=> How are you tiny? I am small. We are the same size.
We are both tiny.

This is a simple dialogue between two entities, one of which describes themselves as "tiny" and the other agrees in size. The conclusion drawn from this exchange is that both entities are indeed tiny. However, the problem statement does not provide specific details about what these entities are or what they do. For the sake of a clear
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2

In [5]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print("Total parameters (including frozen 4 bit) :",total_params)

Total parameters (including frozen 4 bit) : 1698672640


In [6]:

# Using 'train_sft' split as the 'train' split does not exist in this dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00000-of-00003-a3ecf92756(…): downloading bytes:           |  0.00B            

data/train_sft-00001-of-00003-0a1804bcb6(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00001-of-00003-0a1804bcb6(…): downloading bytes:           |  0.00B            

data/train_sft-00002-of-00003-ee46ed25cf(…): reconstructing file:   0%|          |  0.00B /  244MB            

... (12 outputs left out)

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'prompt_id', 'messages'],
    num_rows: 207865
})

In [7]:
def generate_prompt(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

def formatting_func(example):
    return {"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)


Map:   0%|          | 0/207865 [00:00<?, ? examples/s]

In [8]:
dataset[0]

{'prompt': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!\nYour Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.\nDoes this feature apply to all sections of the theme or just specific ones as listed in the text material?",
 'prompt_id': 'f0e37e9f7800261167ce91143f98f511f768847236f133f2d0aed60b444ebe57',
 'messages': [{'content': "These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?\nOn your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of 

In [9]:
dataset = dataset.select(range(7000))

In [10]:
dataset = dataset.shuffle(seed=42)

In [11]:
dataset

Dataset({
    features: ['prompt', 'prompt_id', 'messages', 'text'],
    num_rows: 7000
})

In [ ]:
# Full fine-tuning
# Cheap fine-tuning
# PEFT = Parameter Efficient Fine Tuning
# OOM = Out Of Memory

In [22]:

lora_config = LoraConfig(
    r=16, # rank
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [16]:
model = get_peft_model(model,lora_config)

In [17]:
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
# QLora
# Lora